<a href="https://colab.research.google.com/github/mhtjsh/SedPuzzle/blob/Primary/Puzzle_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Dataset generator for sed-style puzzles. (with distractors)
----------------------------------------

The puzzles are created using a backward-generation strategy:
- Start from an empty string, insert random "src" pieces.
- Record which transitions were used.
- Build the shortest solution as the reverse order of insertions.
- Add distractor transitions for difficulty.
- Occasionally create "multi-tgt" puzzles where a single src has 3–4 tgt rules.
"""

import random
import string
from pathlib import Path

from schema import Problem, Transition, Solution
from utils import write_problem_folder, write_solution_folder

# Allowed characters for src/tgt generation
CHARSET = list(string.ascii_letters + string.digits + ".#@_")


def random_src(min_len=1, max_len=5):
    """Generate a random non-empty src string."""
    return ''.join(random.choices(CHARSET, k=random.randint(min_len, max_len)))


def random_tgts_for_src(src, max_tgts=None):
    """
    Generate tgt options for a given src.
    - If src is long (>3), default is 3 tgt options.
    - Otherwise, default is 2.
    - At least one tgt will be "".
    """
    if max_tgts is None:
        max_tgts = 3 if len(src) > 3 else 2

    tgts = []
    while len(tgts) < max_tgts:
        if random.random() < 0.35:
            candidate = ""
        else:
            length = random.randint(0, min(3, len(src)))
            candidate = ''.join(random.choices(CHARSET, k=length))
        if candidate not in tgts:
            tgts.append(candidate)

    # Ensure one tgt is ""
    if "" not in tgts:
        tgts[random.randrange(len(tgts))] = ""

    return tgts


def generate_problem(problem_id, difficulty="medium", multi_tgt=False):
    """
    Generate a single puzzle + solution.
    - If multi_tgt=True, enforce 3–4 tgt options for one chosen src.
    - Otherwise, use standard backward generation.
    """

    # Step count depends on difficulty
    steps = {
        "easy": random.randint(3, 5),
        "medium": random.randint(6, 10),
        "hard": random.randint(11, 15),
    }[difficulty]

    cur = ""              # start from empty
    used_pairs = []       # keep (src, tgt) for transitions used

    # Build the initial string backwards
    for _ in range(steps):
        src = random_src()
        pos = random.randint(0, len(cur))
        cur = cur[:pos] + src + cur[pos:]
        used_pairs.append((src, ""))  # tgt is "" for removals

    distractors = []
    if multi_tgt:
        # Choose one src to have multiple tgt rules
        src = random_src()
        tgts = random_tgts_for_src(src, max_tgts=random.choice([3, 4]))
        for tgt in tgts:
            distractors.append((src, tgt))
    else:
        # Number of distractors by difficulty
        dcount = {
            "easy": random.randint(0, 2),
            "medium": random.randint(1, 3),
            "hard": random.randint(2, 5),
        }[difficulty]
        for _ in range(dcount):
            src = random_src()
            for tgt in random_tgts_for_src(src):
                distractors.append((src, tgt))

    combined = used_pairs + distractors

    # Deduplicate while preserving order
    seen = set()
    final_pairs = []
    for pair in combined:
        if pair not in seen:
            final_pairs.append(pair)
            seen.add(pair)

    # Map (src,tgt) to transition index
    index_map = {pair: idx for idx, pair in enumerate(final_pairs)}

    # Solution is the reverse order of used_pairs
    solution_indices = [index_map[p] for p in reversed(used_pairs)]

    # Create Pydantic models
    pid = str(problem_id).zfill(3)
    transitions = [Transition(src=src, tgt=tgt) for (src, tgt) in final_pairs]
    problem = Problem(problem_id=pid, initial_string=cur, transitions=transitions)
    solution = Solution(problem_id=pid, solution=solution_indices)

    return problem, solution


def generate_dataset(n=100):
    """Generate a full dataset with a mix of easy/medium/hard + some multi-tgt puzzles."""
    problems = {}
    solutions = {}
    difficulties = ["easy", "medium", "hard"]

    for i in range(1, n + 1):
        difficulty = random.choice(difficulties)
        multi_tgt = (random.random() < 0.2)  # ~20% are multi-tgt puzzles
        p, s = generate_problem(i, difficulty=difficulty, multi_tgt=multi_tgt)
        problems[p.problem_id] = p
        solutions[s.problem_id] = s

    return problems, solutions


if __name__ == "__main__":
    # New dataset folder
    puzzle_dir = Path("dataset/puzzles")
    solution_dir = Path("dataset/solutions")
    puzzle_dir.mkdir(parents=True, exist_ok=True)
    solution_dir.mkdir(parents=True, exist_ok=True)

    problems, solutions = generate_dataset(100)

    # Save all problems and solutions
    write_problem_folder(problems, path=puzzle_dir)
    write_solution_folder(solutions, path=solution_dir)

    print("✅ Generated 100 puzzles and solutions")
    print(f"  Problems saved in   {puzzle_dir.resolve()}")
    print(f"  Solutions saved in  {solution_dir.resolve()}")


✅ Generated 100 puzzles and solutions
  Problems saved in   /content/dataset/puzzles
  Solutions saved in  /content/dataset/solutions


In [ ]:
"""
Hard multi-token puzzle generator.(with distractors)
-----------------------------------------------

Each puzzle:
- Uses 2–4 distinct tokens (letters/digits/special chars).
- Each token has 3–4 tgt options (at least one is "").
- Initial string built backward by inserting tokens.
- Solution = reverse of insertions (shortest).
"""

import random
import string
from pathlib import Path

from schema import Problem, Transition, Solution
from utils import write_problem_folder, write_solution_folder

# Token pool: letters + digits + special
CHARSET = list(string.ascii_letters + string.digits + "#@._$%&*-")


def choose_tokens():
    """Pick 2–4 distinct single-character tokens."""
    return random.sample(CHARSET, random.randint(2, 4))


def random_tgts_for_token(tokens, max_tgts=None):
    """
    Generate tgt options for a token.
    - 3–4 total.
    - At least one is "".
    - Others are short combos of tokens.
    """
    if max_tgts is None:
        max_tgts = random.choice([3, 4])

    tgts = []
    while len(tgts) < max_tgts:
        if random.random() < 0.3:
            candidate = ""
        else:
            length = random.randint(1, 2)
            candidate = ''.join(random.choices(tokens, k=length))
        if candidate not in tgts:
            tgts.append(candidate)

    if "" not in tgts:
        tgts[random.randrange(len(tgts))] = ""

    return tgts


def generate_problem(problem_id):
    """Generate one puzzle using backward insertion with multi-tgt tokens."""
    tokens = choose_tokens()

    # Build initial string backward
    cur = ""
    used_pairs = []
    steps = random.randint(8, 15)  # harder puzzles

    for _ in range(steps):
        src = random.choice(tokens)
        pos = random.randint(0, len(cur))
        cur = cur[:pos] + src + cur[pos:]
        used_pairs.append((src, ""))

    # Build transitions: each token gets 3–4 tgts
    transitions = []
    for token in tokens:
        for tgt in random_tgts_for_token(tokens):
            transitions.append((token, tgt))

    # Deduplicate
    seen = set()
    final_pairs = []
    for pair in transitions:
        if pair not in seen:
            final_pairs.append(pair)
            seen.add(pair)

    # Map (src,tgt) to transition index
    index_map = {pair: idx for idx, pair in enumerate(final_pairs)}

    # Solution = reverse of used_pairs
    solution_indices = [index_map[p] for p in reversed(used_pairs)]

    # Wrap into schema
    pid = str(problem_id).zfill(3)
    problem = Problem(
        problem_id=pid,
        initial_string=cur,
        transitions=[Transition(src=src, tgt=tgt) for (src, tgt) in final_pairs],
    )
    solution = Solution(problem_id=pid, solution=solution_indices)

    return problem, solution


def generate_dataset(n=20):
    """Generate exactly n puzzles."""
    problems, solutions = {}, {}
    for i in range(1, n + 1):
        p, s = generate_problem(i)
        problems[p.problem_id] = p
        solutions[s.problem_id] = s
    return problems, solutions


if __name__ == "__main__":
    puzzle_dir = Path("dataset_multi_token/puzzles")
    solution_dir = Path("dataset_multi_token/solutions")
    puzzle_dir.mkdir(parents=True, exist_ok=True)
    solution_dir.mkdir(parents=True, exist_ok=True)

    problems, solutions = generate_dataset(20)

    write_problem_folder(problems, path=puzzle_dir)
    write_solution_folder(solutions, path=solution_dir)

    print("✅ Generated 20 multi-token puzzles and solutions")
    print(f"  Problems saved in   {puzzle_dir.resolve()}")
    print(f"  Solutions saved in  {solution_dir.resolve()}")


✅ Generated 20 multi-token puzzles and solutions
  Problems saved in   /content/dataset_multi_token/puzzles
  Solutions saved in  /content/dataset_multi_token/solutions


In [ ]:
"""
Fixed generator: Hard sed-style puzzles with transformation chains (50 puzzles).

- Tokens are atomic (strings length 1-3).
- Transitions map tokens -> list_of_tokens internally (empty list means deletion).
- For JSON output we store tgt as the concatenated string (same as previous format).
- Simulation uses token lists (no character-level splitting) so KeyError is avoided.
"""

import random
import string
from pathlib import Path

from schema import Problem, Transition, Solution
from utils import write_problem_folder, write_solution_folder

# Token character pool
CHARSET = list(string.ascii_letters + string.digits + "#@._$%&*-+")


def random_token():
    """Return a random token (1..3 characters)."""
    return ''.join(random.choices(CHARSET, k=random.randint(1, 3)))


def build_transitions(tokens):
    """
    Build transitions for a given list of unique tokens.

    Returns:
      transitions: list of (src, tgt_str) for saving to JSON (tgt_str = '' for deletion),
      token_map: dict src -> list_of_tokens (empty list for deletion) for simulation.
    """
    chain = tokens.copy()
    random.shuffle(chain)

    transitions = []
    token_map = {}

    for i, src in enumerate(chain):
        if i == len(chain) - 1:
            # last token vanishes
            tgt_list = []
            tgt_str = ""
        else:
            nxt = chain[i + 1]
            # either convert to the next token once, or to two copies of next token
            if random.random() < 0.5:
                tgt_list = [nxt]
            else:
                tgt_list = [nxt, nxt]
            tgt_str = ''.join(tgt_list)

        token_map[src] = tgt_list
        transitions.append((src, tgt_str))

    return transitions, token_map


def generate_problem(problem_id):
    """
    Generate one puzzle:
    - choose 3-5 tokens
    - build transition chain (some tokens convert, last -> "")
    - build initial token sequence (8-12 tokens)
    - simulate left-to-right transformation using token_map to get solution indices
    """
    # choose unique tokens
    tokens = []
    while len(tokens) < random.randint(3, 5):
        t = random_token()
        if t not in tokens:
            tokens.append(t)

    # build transitions and internal token_map (list-of-tokens)
    transitions, token_map = build_transitions(tokens)

    # build initial token sequence (list of token-strings)
    token_seq = [random.choice(tokens) for _ in range(random.randint(8, 12))]
    initial_string = ''.join(token_seq)

    # create mapping (src, tgt_str) -> index for solution encoding
    index_map = {(src, tgt_str): idx for idx, (src, tgt_str) in enumerate(transitions)}

    # simulate step-by-step using token lists (not characters)
    cur_tokens = token_seq.copy()
    solution = []
    while cur_tokens:
        left = cur_tokens.pop(0)             # leftmost token (atomic)
        tgt_list = token_map[left]           # list of tokens (maybe empty)
        tgt_str = ''.join(tgt_list)          # string form used in transitions list
        # append the index of the (left -> tgt_str) transition
        solution.append(index_map[(left, tgt_str)])
        # if tgt_list non-empty, insert those tokens at front (they appear where left was)
        if tgt_list:
            cur_tokens = tgt_list + cur_tokens

    # build Pydantic objects for output
    pid = str(problem_id).zfill(3)
    transitions_objs = [Transition(src=src, tgt=tgt_str) for src, tgt_str in transitions]
    problem = Problem(problem_id=pid, initial_string=initial_string, transitions=transitions_objs)
    solution_obj = Solution(problem_id=pid, solution=solution)
    return problem, solution_obj


def generate_dataset(n=50):
    problems = {}
    solutions = {}
    for i in range(1, n + 1):
        p, s = generate_problem(i)
        problems[p.problem_id] = p
        solutions[s.problem_id] = s
    return problems, solutions


if __name__ == "__main__":
    out_p = Path("traverse_replacing_str_puzzles/puzzles")
    out_s = Path("traverse_replacing_str_puzzles/solutions")
    out_p.mkdir(parents=True, exist_ok=True)
    out_s.mkdir(parents=True, exist_ok=True)

    problems, solutions = generate_dataset(50)

    write_problem_folder(problems, path=out_p)
    write_solution_folder(solutions, path=out_s)

    print("\n✅ Done — 50 puzzles written to:")
    print("  ", out_p.resolve())
    print("  ", out_s.resolve())


Sample problem (first):
{
  "problem_id": "001",
  "initial_string": "eehEe_zyyhEehEey_zy",
  "transitions": [
    {
      "src": "_z",
      "tgt": "hEehEe"
    },
    {
      "src": "hEe",
      "tgt": "y"
    },
    {
      "src": "y",
      "tgt": "ee"
    },
    {
      "src": "e",
      "tgt": ""
    }
  ]
}

Sample solution (first):
{
  "problem_id": "001",
  "solution": [
    3,
    3,
    1,
    2,
    3,
    3,
    0,
    1,
    2,
    3,
    3,
    1,
    2,
    3,
    3,
    2,
    3,
    3,
    2,
    3,
    3,
    1,
    2,
    3,
    3,
    1,
    2,
    3,
    3,
    2,
    3,
    3,
    0,
    1,
    2,
    3,
    3,
    1,
    2,
    3,
    3,
    2,
    3,
    3
  ]
}

✅ Done — 50 puzzles written to:
   /content/traverse_replacing_str_puzzles/puzzles
   /content/traverse_replacing_str_puzzles/solutions
